# 5단계: 로컬 베이스 모델 + RAG 예측 생성 (Colab)

목표: Colab T4(16GB)에서 vLLM으로 Qwen2.5-Coder-7B-Instruct(AWQ)를 서빙해 조건 3의 예측을 생성한다.
조건 2와 모델·샘플링 설정이 완전히 동일하고, **프롬프트만 다르다**.

검색(few-shot)과 schema linking은 로컬에서 이미 끝났다 -- `scripts/build_rag_prompts.py`가 만든
`prompts.jsonl`을 여기서는 그대로 재생하기만 한다. 두 변형을 각각 측정한다:

| 변형 | 프롬프트 파일 | 내용 |
|---|---|---|
| `local_rag` | `data/results/local_rag/prompts.jsonl` | few-shot 예제 5개 + 전체 스키마 |
| `local_rag_linked` | `data/results/local_rag_linked/prompts.jsonl` | 같은 few-shot 예제 5개 + 상위 5개 테이블로 축소한 스키마 |
| `local_rag_values` | `data/results/local_rag_values/prompts.jsonl` | 같은 few-shot 예제 5개 + 전체 스키마 + 질문에 등장하는 DB 실제 값 |
| `local_rag_values_only` | `data/results/local_rag_values_only/prompts.jsonl` | few-shot 없이, 전체 스키마 + DB 실제 값만 |
| `local_rag_values_only_v2` | `data/results/local_rag_values_only_v2/prompts.jsonl` | 같은 구성, 값 매처 개선판 |

앞의 세 변형은 few-shot 예제가 전부 동일하므로 차이가 각각 스키마 축소와 값 주입에서만 온다.
네 번째 변형은 예제를 빼서 조건 2/3a/3d/3c가 (few-shot × 값)의 2×2가 되게 한다 —
값이 안 붙은 643개는 조건 2의 프롬프트와 바이트 단위로 같다.

`_v2`는 값 매처를 고친 뒤(굴절어미·문장부호로 끝나는 값·붙여쓴 값) 다시 만든 3d 프롬프트다.
gold 리터럴 리콜이 81.3% → 88.4%로 올랐고 3d와 76개 프롬프트가 다르다. 값이 안 붙은 예제는
624개로, 이쪽도 조건 2와 바이트 단위로 같아 대조군 역할을 그대로 한다.

채점(test-suite accuracy)은 `test_suite_database`(4.9GB, 로컬에만 있음)가 필요해 노트북에서 하지 않는다.

**사전 준비**: 런타임 유형을 GPU(T4)로 설정해두었는지 확인.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q vllm
!pip uninstall -y -q torchaudio
!pip install -q -U torchvision
!pip install -q git+https://github.com/MumuKim0212/text2sql-rag-vs-finetune.git

## 프롬프트 파일 업로드

아래 `VARIANT`를 측정할 변형으로 맞춘 뒤, 해당 폴더의 `prompts.jsonl`을 업로드한다.
업로드된 파일은 `{VARIANT}_prompts.jsonl`로 이름을 바꿔 보관하므로, 한 세션에서 두 변형을 연달아 돌려도 섞이지 않는다.

**GPU 할당량 절약**: 두 변형을 한 세션에서 처리하려면 첫 변형이 끝난 뒤 `VARIANT`만 바꾸고
이 셀부터 아래로 다시 실행하면 된다. 모델 로드 셀은 엔진이 이미 있으면 건너뛰므로 같이 다시 실행해도
안전하다 -- 그냥 재실행하면 첫 엔진이 VRAM의 85%를 쥔 채로 두 번째 엔진을 만들다 실패한다.

In [ ]:
from pathlib import Path

VARIANT = "local_rag"  # local_rag_linked / local_rag_values / local_rag_values_only[_v2]

PROMPTS_PATH = Path(f"{VARIANT}_prompts.jsonl")
OUT_PATH = Path(f"qwen_{VARIANT}_dev_predictions.jsonl")

if not PROMPTS_PATH.exists():
    from google.colab import files

    uploaded = files.upload()  # data/results/{VARIANT}/prompts.jsonl 선택
    assert "prompts.jsonl" in uploaded, "prompts.jsonl을 업로드해야 함"
    Path("prompts.jsonl").rename(PROMPTS_PATH)

print("prompts:", PROMPTS_PATH, "-> predictions:", OUT_PATH)

In [ ]:
import os

# Jupyter/Colab: vLLM's suppress_stdout() calls sys.stdout.fileno(), which
# ipykernel's stdout doesn't support -- but vLLM skips that call entirely
# when VLLM_LOGGING_LEVEL=DEBUG, so this sidesteps the crash.
os.environ["VLLM_LOGGING_LEVEL"] = "DEBUG"
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

from vllm import LLM, SamplingParams

MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct-AWQ"

# Skip if an engine is already up: vLLM reserves gpu_memory_utilization of VRAM
# at construction, so building a second one here fails before the first can be
# freed -- which is what happens when this cell is re-run to do the other
# variant. Both variants use this same model, so reusing the engine is correct.
if "llm" in globals():
    print("engine already loaded, reusing:", MODEL_ID)
else:
    llm = LLM(
        model=MODEL_ID,
        quantization="awq",
        dtype="float16",
        gpu_memory_utilization=0.85,
        max_model_len=4096,
    )
    print("loaded:", MODEL_ID)

## 스모크 테스트

RAG 프롬프트는 조건 2보다 길다(최대 ~4,400자 ≈ 1,100토큰). `max_model_len=4096`에 여유가 있는지
가장 긴 프롬프트로 먼저 확인한다.

In [ ]:
import json

from rag_text2sql.models.cloud import SYSTEM_PROMPT

prompts = [json.loads(line) for line in PROMPTS_PATH.open(encoding="utf-8") if line.strip()]
print(f"{len(prompts)} prompts, max {max(len(p['prompt']) for p in prompts)} chars")

longest = max(prompts, key=lambda p: len(p["prompt"]))
smoke_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": longest["prompt"]},
]

smoke_out = llm.chat([smoke_messages], SamplingParams(temperature=0.0, max_tokens=256))
print("gold:", longest["gold_sql"])
print("pred:", smoke_out[0].outputs[0].text.strip())

## Spider dev 전체 생성

조건 2와 동일하게 32개 단위로 배치 처리하며 배치마다 append + flush -- 세션이 끊겨도 이미 쓴 줄은 건너뛰고 재개된다.
샘플링 파라미터도 조건 2와 같다(`temperature=0.0, max_tokens=256`).

In [ ]:
import time

import torch

BATCH_SIZE = 32

done = []
if OUT_PATH.exists():
    with OUT_PATH.open(encoding="utf-8") as f:
        done = [json.loads(line) for line in f if line.strip()]
start = len(done)
print(f"Resuming: {start} examples already done" if start else "Starting fresh")

sampling_params = SamplingParams(temperature=0.0, max_tokens=256)

# The brief asks for latency and VRAM alongside accuracy, and only the GPU run
# can see them. Peak memory is reset after the engine is built, so what is
# reported is generation's own high-water mark on top of the loaded weights.
# Prompt length is what varies across these variants -- 3d's are about half of
# 3c's, which was one of the reasons condition 5 was built on 3d, argued so far
# rather than measured.
torch.cuda.reset_peak_memory_stats()
batch_seconds = []
run_started = time.perf_counter()

with OUT_PATH.open("a", encoding="utf-8") as f:
    for batch_start in range(start, len(prompts), BATCH_SIZE):
        batch = prompts[batch_start : batch_start + BATCH_SIZE]
        conversations = [
            [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": ex["prompt"]},
            ]
            for ex in batch
        ]
        batch_started = time.perf_counter()
        outputs = llm.chat(conversations, sampling_params)
        batch_seconds.append(time.perf_counter() - batch_started)
        for ex, out in zip(batch, outputs):
            record = {
                "question": ex["question"],
                "db_id": ex["db_id"],
                "gold_sql": ex["gold_sql"],
                "pred_sql": out.outputs[0].text.strip(),
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()
        print(f"[{min(batch_start + BATCH_SIZE, len(prompts))}/{len(prompts)}]")

print("done:", OUT_PATH)

elapsed = time.perf_counter() - run_started
measured = len(prompts) - start
runtime = {
    "variant": VARIANT,
    "gpu": torch.cuda.get_device_name(0),
    "adapter": False,
    "batch_size": BATCH_SIZE,
    # A resumed run only times what it generated, so record that count, not len(prompts).
    "measured_examples": measured,
    "total_seconds": round(elapsed, 1),
    "seconds_per_example": round(elapsed / measured, 3),
    "examples_per_second": round(measured / elapsed, 2),
    "median_batch_seconds": round(sorted(batch_seconds)[len(batch_seconds) // 2], 2),
    "median_prompt_chars": sorted(len(p["prompt"]) for p in prompts)[len(prompts) // 2],
    "generation_peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
    "peak_reserved_vram_gb": round(torch.cuda.max_memory_reserved() / 1e9, 2),
}
RUNTIME_PATH = Path(f"{VARIANT}_runtime.json")
RUNTIME_PATH.write_text(json.dumps(runtime, indent=2), encoding="utf-8")
print(json.dumps(runtime, indent=2))

## 예측 파일 다운로드

다운로드한 파일을 로컬 저장소의 `data/results/{VARIANT}/`에 넣고 아래로 채점:

```bash
uv run python scripts/score_predictions.py --predictions data/results/local_rag/qwen_local_rag_dev_predictions.jsonl --condition local_rag
uv run python scripts/score_predictions.py --predictions data/results/local_rag_linked/qwen_local_rag_linked_dev_predictions.jsonl --condition local_rag_linked
uv run python scripts/score_predictions.py --predictions data/results/local_rag_values/qwen_local_rag_values_dev_predictions.jsonl --condition local_rag_values
uv run python scripts/score_predictions.py --predictions data/results/local_rag_values_only/qwen_local_rag_values_only_dev_predictions.jsonl --condition local_rag_values_only
uv run python scripts/score_predictions.py --predictions data/results/local_rag_values_only_v2/qwen_local_rag_values_only_v2_dev_predictions.jsonl --condition local_rag_values_only_v2
```

채점은 1,034개에 약 40초 걸린다.

In [ ]:
from google.colab import files

files.download(str(OUT_PATH))
files.download(str(RUNTIME_PATH))  # latency/VRAM -> data/results/{VARIANT}/runtime.json